# Continuation of traveling waves of the OV model

## Description

This Notebook is a supplemental material of the manuscript "" by K. ov and T. Miyaji.

In [1]:
import numpy as np
import numpy.random as rd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.linalg as la
from scipy.fft import rfft, irfft, rfftfreq 
from scipy.optimize import fsolve, root
from scipy.interpolate import CubicSpline
from scipy.integrate import quad
import numpy.random as rd
%matplotlib inline
sns.set('notebook', 'whitegrid', 'dark', font_scale=2, rc={"lines.linewidth": 2, 'grid.linestyle': '-'})

## Subroutines

The OV function and its derivative

In [2]:
def V(x, bp_V0, bp_beta, bp_lc, bp_M):
    return bp_V0 * (np.tanh(bp_beta * (x - bp_lc)) + bp_M)

def dV(x, bp_V0, bp_beta, bp_lc, bp_M):
    return bp_beta * bp_V0 / np.cosh(bp_beta*(x-bp_lc))**2

Characterization of a TW solution

In [3]:
def func7(u, *args):
    d = 2
    bp_beta, a, L, bp_V0, bp_lc, bp_M, N, n, p, u0 = args
    f = np.zeros(1+n//2, dtype=complex)
    f[0] = L/N + 0.j
    f[1:] = u[0:-1:2] + 1.j * u[1:-1:2]
    c = u[-1]
    d1 = np.array([2.j*np.pi*m/N for m in range(len(f))])
    d2 = d1 * d1
    F  = irfft(f, n=d*n, norm="forward")
    B1 = V(np.roll(F, -d*p), bp_V0, bp_beta, bp_lc, bp_M) - V(F, bp_V0, bp_beta, bp_lc, bp_M)
    B = -c * (c*d2 + a*d1) * f + a * rfft(B1, n=d*n, norm="forward")[:n//2+1] 
    ans = np.zeros(n+1)
    ans[:-1:2]  = np.real(B[1:])
    ans[1:-1:2] = np.imag(B[1:])

    df0 = np.zeros(1+n//2, dtype=complex)
    # 導関数
    df0[1:] = u0[0:-1:2] + 1.j*u0[1:-1:2]
    # df0 = np.array([ -2.j * np.pi * k * df0[k] / L for k in range(len(df0))])
    df0 = np.array([ -2.j * np.pi * k * df0[k] / N for k in range(len(df0))])
    dF0  = irfft(df0, n=d*n, norm="forward")
    # 積分
    ans[-1] = (F @ dF0) / N
    return ans

Jacobian matrix of the above function

In [4]:
def dfunc7(u, *args):
    bp_beta, a, L, bp_V0, bp_lc, bp_M, N, n, p, u0 = args
    eps = 2**(-10)
    ans = np.zeros((len(u), len(u)))
    for k in range(len(u)):
        e = np.zeros(len(u))
        e[k] = 1.0
        df1 = (func7(u + eps * e, *args) - func7(u - eps * e, *args)) / (2 * eps)
        df2 = (func7(u + 2*eps * e, *args) - func7(u - 2*eps * e, *args)) / (4 * eps)
        ans[:,k] = (df1 - 0.25*df2) / 0.75
    return ans

Derivative of func7 above w.r.t. a parameter

In [5]:
def dfda7(u, *args):
    eps = 2**(-10)
    bp_beta, a, L, bp_V0, bp_lc, bp_M, N, n, p, u0 = args
    args2 = list(args)
    args2[0] = a + eps
    df11 = func7(u, *tuple(args2))
    args2[0] = a - eps
    df12 = func7(u, *tuple(args2))
    df1 = (df11 - df12) / (2*eps)
    
    args2[0] = a + 2*eps
    df11 = func7(u, *tuple(args2))
    args2[0] = a - 2*eps
    df12 = func7(u, *tuple(args2))
    df2 = (df11 - df12) / (4*eps)
    
    return (df1 - df2 * 0.25) / 0.75

Corrector

In [6]:
def correct(v, ut, a, L, bp_V0, bp_lc, bp_M, N, n, p, y0,tol):

    maxiter = 10
    eps = tol

    def func(u, *args):
        args2 = list(args)
        args2.insert(0, u[-1])
        F = np.zeros(len(u))
        F[:-1] = func7(u[:-1], *tuple(args2))
        F[-1] = (u - ut)@v
        return F

    def dfunc(u, *args):
        args2 = list(args)
        args2.insert(0, u[-1])
        DF = np.zeros((len(u), len(u)))
        DF[:-1,:-1] = dfunc7(u[:-1], *tuple(args2))
        DF[:-1,-1] = dfda7(u[:-1], *tuple(args2))
        DF[-1, :] = v
        return DF
    
    args=(a, L, bp_V0, bp_lc, bp_M, N, n, p, y0)
    z = np.copy(ut)
    b = func(z, *args)
    for m in range(maxiter):
        A = dfunc(z, *args)
        eta = la.solve(A, b) 
        z -= eta
        b = func(z, *args)
        res = la.norm(b, ord=np.inf)
        if res * (1 + 1.0/la.norm(z, ord=np.inf)) < eps or la.norm(eta, ord=np.inf) / la.norm(z, ord=np.inf) < eps:
            return z, res, m, True
    return z, res, maxiter, False

Predictor

In [7]:
def predict(u0, v0, A, b, h, a, L, bp_V0, bp_lc, bp_M, N, n, p):
    args = (u0[-1], a, L, bp_V0, bp_lc, bp_M, N, n, p, u0[:-1])

    J = np.zeros((len(u0), len(u0)))
    J[:-1, :-1] = A
    J[:-1, -1] = b
    J[-1, :] = v0

    e = np.zeros(len(u0))
    e[-1] = 1.0

    v = la.solve(J, e)
    v = v / la.norm(v)
    return v, u0 + h*v, la.norm(e - J@v), v@v0

Solver of a bordering system
$$
\begin{pmatrix}
A & \vec{b} \\
\vec{c}^T & d
\end{pmatrix} w = \begin{pmatrix} \vec{0} \\ 1 \end{pmatrix}
$$

In [8]:
def bodering(A, b, c, d):
    M = np.zeros((len(c)+1, len(c)+1))
    M[:-1,:-1] = A
    M[-1,:-1] = c
    M[:-1,-1] = b
    M[-1,-1] = d
    e = np.zeros(len(c)+1) # e = (0, 0, ..., 1)
    e[-1] = 1.0
    w = la.solve(M, e)
    return w

Location of a limit point

In [9]:
def locateLP(y, v, a, L, bp_V0, bp_lc, bp_M, N, n, p, y0, eps=1.0e-8, h=2**(-26), maxiter=10):

    e = np.zeros(len(y)) # e = (0, 0, ..., 1)
    e[-1] = 1.0

    z = np.copy(y)
    args=(a, L, bp_V0, bp_lc, bp_M, N, n, p, y0)
    args2 = list(args)
    args2.insert(0, z[-1])

    d = dfda7(z[:-1], *tuple(args2))
    c = v[:-1]

    for m in range(maxiter):
        x0 = z[:-1]
        a0 = z[-1]
        args2[0] = a0

        b = np.zeros(len(y))
        b[:-1] = func7(x0, *tuple(args2))
        
        M = np.zeros((len(y), len(y)))
        M[:-1,:-1] = dfunc7(x0, *tuple(args2))
        M[-1,:-1] = c
        M[:-1,-1] = d
        w = la.solve(M, e)
        b[-1] = w[-1]
        if la.norm(b, ord=np.inf) < eps:
            return z, b[-1], m

        phi = la.solve(M.T, e)
        A = np.zeros((len(y), len(y)))
        A[:-1,:-1] = M[:-1, :-1]
        A[:-1,-1] = dfda7(x0, *tuple(args2))
        A[-1,:-1] = -phi[:-1] @ (dfunc7(x0+h*w[:-1], *tuple(args2)) - dfunc7(x0-h*w[:-1], *tuple(args2))) / (2*h)
        args2[0] = a0 + h
        d1 = dfunc7(x0, *tuple(args2))
        args2[0] = a0 - h
        d2 = dfunc7(x0, *tuple(args2))
        A[-1,-1] = -phi[:-1] @ ((d1 - d2) / (2*h)) @ w[:-1]
        z -= la.solve(A, b)
    print("not converged")

Path-following by predictor-corrector method

In [10]:
# 解の追跡バージョン7
def pathfollow07(y0, a0, vel0, a, L, bp_V0, bp_lc, bp_M, N, p, nmax=100, intv=5, h=1e-2, hmin=1e-5, hmax=1e-2, n_f=2, n_s=2, tol=1e-12):
    fac_i = 2.0
    fac_d = 0.25

    bd = []
    u0=np.zeros(len(y0)+1)
    u0[:-1] = y0
    u0[-1] = a0
    print("# rho", "Re(f0)", "c", "h", "|b-Jv|", "(v,v0)", "residual", "num_iter")
    v = vel0[:]

    J = dfunc7(u0[:-1], u0[-1], a, L, bp_V0, bp_lc, bp_M, N, p*N, p, u0[:-1])
    fa = dfda7(u0[:-1], u0[-1], a, L, bp_V0, bp_lc, bp_M, N, p*N, p, u0[:-1])
    v, ut, res_predictor, ip_predictor = predict(u0, v, J, fa, h, a, L, bp_V0, bp_lc, bp_M, N, p*N, p)
    # Test function for a limit point
    flp = bodering(J, fa, v[:-1], 0.0)
    flp0 = flp[-1]

    # Test function for a bifurcation point
    # B = np.zeros((len(y0)+1, len(y0)+1))
    # B[:-1,:-1] = J
    # B[:-1,-1] = fa
    # B[-1,:]=v
    # fbp0 = la.det(B)

    # bd.append({'TY':"R", 'x':u0[:-1], 'a':u0[-1], 'v':v, 'residual':0.0, 'ds':h, 'num_iter':0, 'flp':flp0, 'fbp': fbp0})
    bd.append({'TY':"R", 'x':u0[:-1], 'a':u0[-1], 'v':v, 'residual':0.0, 'ds':h, 'num_iter':0, 'flp':flp0})
    n = 1
#     for n in range(1, nmax+1):
    while n < nmax+1:
        flg = False
        while flg is False:
            w, res, nitr, flg = correct(v, ut, a, L, bp_V0, bp_lc, bp_M, N, p*N, p, u0[:-1],tol)
            if nitr < n_f:
                h = min(fac_i*h, hmax)
            elif nitr > n_s and h > hmin:
                h = max(fac_d*h, hmin)
                ut = u0 + h * v
                flg = False
            elif nitr > n_s and h <= hmin:
                print(n, res, nitr, flg)
                return bd
            else:
                pass

        # Test function for a limit point
        J    = dfunc7(w[:-1], w[-1], a, L, bp_V0, bp_lc, bp_M, N, p*N, p, u0[:-1])
        fa   = dfda7(w[:-1], w[-1], a, L, bp_V0, bp_lc, bp_M, N, p*N, p, u0[:-1])
        flp  = bodering(J, fa, v[:-1], 0.0)
        flp1 = flp[-1]
        
        # Test function for a bifurcation point
        # B = np.zeros((len(y0)+1, len(y0)+1))
        # B[:-1,:-1] = J
        # B[:-1,-1] = fa
        # B[-1,:]=v
        # fbp1 = la.det(B)

        # if fbp0 * fbp1 < 0:
            # print("A bifurcation point is detected")
        # elif flp0 * flp1 < 0:
        if flp0 * flp1 < 0:
            print("A limit point is detected")
            ylp, flp_c, niter_lp = locateLP(u0, v, a, L, bp_V0, bp_lc, bp_M, N, p*N, p, u0[:-1])
            J2    = dfunc7(ylp[:-1], ylp[-1], a, L, bp_V0, bp_lc, bp_M, N, p*N, p, u0[:-1])
            fa2   = dfda7(ylp[:-1], ylp[-1], a, L, bp_V0, bp_lc, bp_M, N, p*N, p, u0[:-1])
            v2, ut2, res_predictor2, ip_predictor2 = predict(ylp, v, J2, fa2, h, L, N, p*N, p)
            # bd.append({'TY':"L", 'x':ylp[:-1], 'a':ylp[-1], 'v':v2, 'residual':res, 'ds':h, 'num_iter':niter_lp, 'flp':flp_c, 'fbp':fbp1})
            bd.append({'TY':"L", 'x':ylp[:-1], 'a':ylp[-1], 'v':v2, 'residual':res, 'ds':h, 'num_iter':niter_lp, 'flp':flp_c})
            print("{0:d} {1:e} {2:e} {3:e}".format(n, ylp[-1], ylp[0], ylp[-2]))
            n = n + 1
        else:
            pass

        u0 = w[:]

        if n%intv==0:
            # print("{0:d} {1:e} {2:e} {3:e} {4:e} {5:e} {6:e} {7:e} {8:d} {9:e} {10:e}".format(n, w[-1], w[0], w[-2], h, res_predictor, ip_predictor, res, nitr, flp1, fbp1))
            print("{0:d} {1:e} {2:e} {3:e} {4:e} {5:e} {6:e} {7:e} {8:d} {9:e}".format(n, w[-1], w[0], w[-2], h, res_predictor, ip_predictor, res, nitr, flp1))

        n = n + 1
        # fbp0 = fbp1
        flp0 = flp1
        v, ut, res_predictor, ip_predictor = predict(u0, v, J, fa, h, a, L, bp_V0, bp_lc, bp_M, N, p*N, p)
        # bd.append({'TY':"R", 'x':w[:-1], 'a':w[-1], 'v':v, 'residual':0.0, 'ds':h, 'num_iter':0, 'flp':flp0, 'fbp': fbp0})
        bd.append({'TY':"R", 'x':w[:-1], 'a':w[-1], 'v':v, 'residual':0.0, 'ds':h, 'num_iter':0, 'flp':flp0})

    return bd

Distance between two functions on an interval (generated by Claude (Anthropic))

In [11]:
def max_norm_difference(f, g, fargs, gargs, tmin, tmax, initial_points=1000, max_iterations=10, tolerance=1e-8, vectorized=False):
    """
    関数f(t)とg(t)の差f-gの最大値ノルム（無限大ノルム）を計算する
    
    Parameters:
    -----------
    f : callable
        関数f(t)
    g : callable
        関数g(t)
    fargs : tuple
        関数fのパラメータ
    gargs : tuple
        関数gのパラメータ
    tmin : float
        区間の下限
    tmax : float
        区間の上限
    initial_points : int
        初期サンプル点数
    max_iterations : int
        最大反復回数
    tolerance : float
        収束判定の許容誤差
    vectorized : bool
        関数f, gがベクトル化されているかどうか
    
    Returns:
    --------
    dict
        結果の辞書
        - 'max_norm': 最大値ノルム
        - 'converged': 収束したかどうか
        - 'iterations': 実行した反復回数
        - 'final_points': 最終的なサンプル点数
        - 'total_evaluations': 総関数評価回数
        - 'argmax': 最大値を与えるtの値
    """
    
    # 適応的数値計算による最大値探索（ベクトル化対応）
    n_points = initial_points
    prev_max = float('-inf')
    prev_t_max = None
    
    # 初回計算
    t_values = np.linspace(tmin, tmax, n_points)
    if vectorized:
        # ベクトル化された関数の場合
        diff_values = np.abs(f(t_values, *fargs) - g(t_values, *gargs))
    else:
        # 非ベクトル化の場合（従来通り）
        diff_values = np.abs(np.array([f(t, *fargs) for t in t_values]) - np.array([g(t, *gargs) for t in t_values]))
    
    max_idx = np.argmax(diff_values)
    current_max = diff_values[max_idx]
    current_t_max = t_values[max_idx]
    
    total_evaluations = n_points
    
    for iteration in range(max_iterations):
        # 収束判定
        if iteration > 0 and abs(current_max - prev_max) < tolerance:
            return {
                'max_norm': current_max,
                'converged': True,
                'iterations': iteration + 1,
                'final_points': len(t_values),
                'total_evaluations': total_evaluations,
                'argmax': current_t_max
            }
        
        prev_max = current_max
        prev_t_max = current_t_max
        
        # 次の反復では点数を倍にする
        if iteration < max_iterations - 1:  # 最後の反復でない場合
            n_new_points = n_points * 2
            t_values_new = np.linspace(tmin, tmax, n_new_points)
            
            # 新しい点のみを特定（奇数インデックス）
            new_indices = np.arange(1, n_new_points, 2)
            t_new = t_values_new[new_indices]
            
            # 新しい点での関数値を計算
            if vectorized:
                # ベクトル化された関数の場合
                diff_new = np.abs(f(t_new, *fargs) - g(t_new, *gargs))
            else:
                # 非ベクトル化の場合
                f_new = np.array([f(t, *fargs) for t in t_new])
                g_new = np.array([g(t, *gargs) for t in t_new])
                diff_new = np.abs(f_new - g_new)
            
            # 新しい点での最大値をチェック
            new_max_idx = np.argmax(diff_new)
            new_max = diff_new[new_max_idx]
            new_t_max = t_new[new_max_idx]
            
            # 全体の最大値を更新
            if new_max > current_max:
                current_max = new_max
                current_t_max = new_t_max
            # 既存の最大値の方が大きい場合は、current_max と current_t_max はそのまま
            
            # 次の反復の準備
            t_values = t_values_new
            n_points = n_new_points
            total_evaluations += len(t_new)
    
    # 最大反復回数に達した場合
    return {
        'max_norm': current_max,
        'converged': False,
        'iterations': max_iterations,
        'final_points': len(t_values),
        'total_evaluations': total_evaluations,
        'argmax': current_t_max
    }

## Parameters and transition layers

In [12]:
bp_V0, bp_beta, bp_lc, bp_M = 0.5 * 0.0336, 2 / 0.0223, 0.025, 0.913
bp_a = 1.6
bp_eta = 2*bp_V0 * 0.79681213002002 / bp_a

$c_{\infty}(\eta)$

In [13]:
def c_infty(x):
    return -bp_a / np.log(1 - bp_a * x / (2*bp_V0))

$u_i(t)$

In [14]:
def ui(t):
    return np.where(t < -1,
                    bp_lc + bp_eta + 2*bp_V0 * np.log(1 - bp_a * bp_eta / (2*bp_V0)) / bp_a,
                    np.where(t > 0,
                             bp_lc + bp_eta - bp_eta * np.exp(-bp_a * t / c_infty(bp_eta)),
                             bp_lc + bp_eta - (2*bp_V0 * (1 - np.exp(-bp_a*t/c_infty(bp_eta))) / bp_a - 2*bp_V0 * t / c_infty(bp_eta) + bp_eta * np.exp(-bp_a*t / c_infty(bp_eta)))))

$u_d(t) = 2l_c - u_i(t)$

In [15]:
def ud(t):
    return 2 * bp_lc - ui(t)

Cutoff function
$$
\xi(x) = \frac{1}{2}\left( 1 - \tanh\left( \frac{1}{x} + \frac{1}{x-1} \right) \right),\quad (0<x<1)
$$

In [16]:
def XI(x):
    result = np.zeros_like(x, dtype=float)
    
    # x > 1の場合
    mask_gt1 = x >= 1
    result[mask_gt1] = 1.0
    
    # x <= 0の場合
    mask_le0 = x <= 0
    result[mask_le0] = 0.0
    
    # 0 < x < 1の場合のみ計算
    mask_middle = (x > 0) & (x < 1)
    if np.any(mask_middle):
        x_middle = x[mask_middle]
        result[mask_middle] = 0.5 * (1.0 - np.tanh(1.0/x_middle + 1.0/(x_middle-1.0)))
    
    return result

def CHI(x):
    return XI(x+2) * (1 - XI(x-1))

$u_0(x)$

In [17]:
def u0(t, u_asterisk, R, N):
    t_asterisk = 0.5 * N * (u_asterisk - bp_lc + bp_eta) / bp_eta
    return np.where(np.abs(t) <= 2*R,
                    ui(t)*CHI(t/R)+(bp_lc-bp_eta)*(1.0 - XI((t+2*R)/R))+(bp_lc+bp_eta)*XI((t-R)/R),
                    np.where(np.abs(t - t_asterisk) <= 2*R,
                             ud(t-t_asterisk)*CHI((t-t_asterisk)/R) + (bp_lc + bp_eta)*(1-XI((t-t_asterisk+2*R)/R))+(bp_lc-bp_eta)*XI((t-t_asterisk-R)/R),
                             np.where((2*R<=t) & (t <= t_asterisk-2*R),
                                      bp_lc + bp_eta,
                                      bp_lc-bp_eta)))

Compute u(x) from Fourier coefficients (generated by Claude (Anthropic))

In [18]:
def u_profile_optimized(x, coef, L, N):
    # 複素数係数を作成
    cf = coef[0:-1:2] + 1.j * coef[1:-1:2]
    
    # k の配列を作成
    k_array = np.arange(1, len(cf)//2)
    
    if np.isscalar(x):
        exp_terms = np.exp(2.j * np.pi * k_array * x / N)
        cf_terms = cf[k_array-1]
        complex_sum = 2 * np.real(np.sum(cf_terms * exp_terms))
    else:
        exp_terms = np.exp(2.j * np.pi * k_array[:, None] * x[None, :] / N)
        cf_terms = cf[k_array-1][:, None]
        complex_sum = 2 * np.real(np.sum(cf_terms * exp_terms, axis=0))
    
    return L/N + complex_sum

In [19]:
def u_profile_optimized_with_shift(x, coef, shift, L, N):
    return u_profile_optimized(x+shift, coef, L, N)

In [20]:
def tobesolved(x, coef, L, N):
    return u_profile_optimized(x, coef, L, N) - bp_lc

## Computation

In [21]:
N20 = 20
n = 1
a = 1.6
p20 = 32
L = 0.5
K=1024

### N=20, L=0.5, p=64

In [22]:
v20_2 = np.load("OV_Bando_N20_L5e-1_p64.npy")
p20_2 = 64

In [ ]:
vel0 = np.zeros(len(v20_2)+1)
vel0[-1] = 1.0
bd20_2_1 = pathfollow07(v20_2, bp_beta, vel0, a, L, bp_V0, bp_lc, bp_M, N20, p20_2, nmax=1500, intv=50, h=1e-3, hmin=1e-5,hmax=10.0, tol=1e-10)

# rho Re(f0) c h |b-Jv| (v,v0) residual num_iter
50 1.675091e+02 1.094296e-02 9.482589e-01 2.048000e+00 4.886004e-10 1.000000e+00 1.216549e-13 2 3.051985e+06
100 2.699092e+02 1.066762e-02 9.792280e-01 2.048000e+00 9.666823e-11 1.000000e+00 9.051755e-13 2 3.898672e+05
150 3.723094e+02 1.054860e-02 9.906334e-01 2.048000e+00 1.955913e-11 1.000000e+00 6.067614e-12 2 2.340026e+05


In [ ]:
bd20_2_2 = pathfollow07(bd20_2_1[-1]['x'], bd20_2_1[-1]['a'], bd20_2_1[-1]['v'],
                      a, L, bp_V0, bp_lc, bp_M, N20, p20_2,
                      nmax=1500, intv=50, h=1.0, hmin=1e-5,hmax=10.0, tol=1e-8)

In [ ]:
bd20_2_3 = pathfollow07(bd20_2_2[-1]['x'], bd20_2_2[-1]['a'], bd20_2_2[-1]['v'],
                      a, L, bp_V0, bp_lc, bp_M, N20, p20_2,
                      nmax=1500, intv=50, h=1.0, hmin=1e-5,hmax=10.0, tol=1e-8)

In [ ]:
bd20_2_4 = pathfollow07(bd20_2_3[-1]['x'], bd20_2_3[-1]['a'], bd20_2_3[-1]['v'],
                      a, L, bp_V0, bp_lc, bp_M, N20, p20_2,
                      nmax=1500, intv=50, h=1.0, hmin=1e-5,hmax=10.0, tol=1e-8)

In [ ]:
bd20_p64 = bd20_2_1 + bd20_2_2 + bd20_2_3 + bd20_2_4

In [ ]:
r = 2.5
MaxDist20 = []
for bd in bd20_p64:
    theta20 = fsolve(tobesolved, 15, args=(bd['x'], 0.5, N20), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta20, 0.5, N20), gargs=(L/N20, r, N20),
                               tmin=-r, tmax=N20-r, vectorized=True)
    MaxDist20.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist20_p64_1 = np.array(MaxDist20)

In [ ]:
np.save('ov_N20_p64_r2.5.npy', MaxDist20_p64_1)

In [ ]:
r = 4.0
MaxDist20 = []
for bd in bd20_p64:
    theta20 = fsolve(tobesolved, 15, args=(bd['x'], 0.5, N20), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta20, 0.5, N20), gargs=(L/N20, r, N20),
                               tmin=-r, tmax=N20-r, vectorized=True)
    MaxDist20.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist20_p64_2 = np.array(MaxDist20)

In [ ]:
np.save('ov_N20_p64_r4.0.npy', MaxDist20_p64_2)

In [ ]:
r = 2.
MaxDist20 = []
for bd in bd20_p64:
    theta20 = fsolve(tobesolved, 15, args=(bd['x'], 0.5, N20), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta20, 0.5, N20), gargs=(L/N20, r, N20),
                               tmin=-r, tmax=N20-r, vectorized=True)
    MaxDist20.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist20_p64_3 = np.array(MaxDist20)

In [ ]:
np.save('ov_N20_p64_r2.0.npy', MaxDist20_p64_3)

In [ ]:
r = 3.
MaxDist20 = []
for bd in bd20_p64:
    theta20 = fsolve(tobesolved, 15, args=(bd['x'], 0.5, N20), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta20, 0.5, N20), gargs=(L/N20, r, N20),
                               tmin=-r, tmax=N20-r, vectorized=True)
    MaxDist20.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist20_p64_4 = np.array(MaxDist20)

In [ ]:
np.save('ov_N20_p64_r3.0.npy', MaxDist20_p64_4)

In [ ]:
r = 1.
MaxDist20 = []
for bd in bd20_p64:
    theta20 = fsolve(tobesolved, 15, args=(bd['x'], 0.5, N20), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta20, 0.5, N20), gargs=(L/N20, r, N20),
                               tmin=-r, tmax=N20-r, vectorized=True)
    MaxDist20.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist20_p64_5 = np.array(MaxDist20)

In [ ]:
np.save('ov_N20_p64_r1.0.npy', MaxDist20_p64_5)

### N=40, L=1.0, p=48

In [56]:
N40 = 40
p40 = 12
p40_4 = 48

In [57]:
n = 1
a = 1.6
L = 1.0
K=1024

In [58]:
v40 = np.load("OV_Bando_N40_L1_p12.npy")

In [60]:
v40_4 = np.zeros(N40*p40_4+1, dtype=float)
v40_4[:N40*p40] = v40[:-1]
v40_4[-1] = v40[-1]

In [61]:
vel0 = np.zeros(len(v40_4)+1)
vel0[-1] = 1.0
tmp = pathfollow07(v40_4, bp_beta, vel0, a, L, bp_V0, bp_lc, bp_M, N40, p40_4, nmax=1, intv=1, h=1e-3, hmin=1e-5,hmax=10.0, tol=1e-12)

# rho Re(f0) c h |b-Jv| (v,v0) residual num_iter
1 8.968710e+01 1.100862e-02 8.822716e-01 2.000000e-03 6.312664e-07 9.999994e-01 4.678030e-15 1 7.920790e+05


In [ ]:
bd40_p48_1 = pathfollow07(tmp[-1]['x'], bp_beta, vel0, a, L, bp_V0, bp_lc, bp_M, N40, p40_4, nmax=2000, intv=50, h=1e-3, hmin=1e-5,hmax=10.0, tol=1e-10, n_f=3, n_s=4)

# rho Re(f0) c h |b-Jv| (v,v0) residual num_iter


In [ ]:
r = 2.5
MaxDist40 = []
for bd in bd40_p48_1:
    theta40 = fsolve(tobesolved, 30, args=(bd['x'], 1.0, N40), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta40, 1.0, N40), gargs=(L/N40, r, N40),
                               tmin=-r, tmax=N40-r, vectorized=True)
    MaxDist40.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist40_p48_1 = np.array(MaxDist40)

In [ ]:
np.save('ov_N40_p48_r2.5.npy', MaxDist40_p48_1)

In [ ]:
r = 4.0
MaxDist40 = []
for bd in bd40_p48_1:
    theta40 = fsolve(tobesolved, 30, args=(bd['x'], 1.0, N40), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta40, 1.0, N40), gargs=(L/N40, r, N40),
                               tmin=-r, tmax=N40-r, vectorized=True)
    MaxDist40.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist40_p48_2 = np.array(MaxDist40)

In [ ]:
np.save('ov_N40_p48_r4.0.npy', MaxDist40_p48_2)

In [ ]:
r = 8.0
MaxDist40 = []
for bd in bd40_p48_1:
    theta40 = fsolve(tobesolved, 30, args=(bd['x'], 1.0, N40), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta40, 1.0, N40), gargs=(L/N40, r, N40),
                               tmin=-r, tmax=N40-r, vectorized=True)
    MaxDist40.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist40_p48_3 = np.array(MaxDist40)

In [ ]:
np.save('ov_N40_p48_r8.0.npy', MaxDist40_p48_3)

In [ ]:
r = 6.0
MaxDist40 = []
for bd in bd40_p48_1:
    theta40 = fsolve(tobesolved, 30, args=(bd['x'], 1.0, N40), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta40, 1.0, N40), gargs=(L/N40, r, N40),
                               tmin=-r, tmax=N40-r, vectorized=True)
    MaxDist40.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist40_p48_4 = np.array(MaxDist40)

In [ ]:
np.save('ov_N40_p48_r6.0.npy', MaxDist40_p48_4)

In [ ]:
r = 2.0
MaxDist40 = []
for bd in bd40_p48_1:
    theta40 = fsolve(tobesolved, 30, args=(bd['x'], 1.0, N40), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta40, 1.0, N40), gargs=(L/N40, r, N40),
                               tmin=-r, tmax=N40-r, vectorized=True)
    MaxDist40.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist40_p48_5 = np.array(MaxDist40)

In [ ]:
np.save('ov_N40_p48_r2.0.npy', MaxDist40_p48_5)

### N=80, L=2.0, p=48

In [90]:
N80 = 80
p80 = 12

In [91]:
n = 1
a = 1.6
L = 2.0
K=1024

In [92]:
v80 = np.load("OV_Bando_N80_L2_p12.npy")

In [93]:
p80_4 = 48
print(p80_4)

48


In [94]:
v80_4 = np.zeros(N80*p80_4+1, dtype=float)
v80_4[:N80*p80] = v80[:-1]
v80_4[-1] = v80[-1]

In [ ]:
vel0 = np.zeros(len(v80_4)+1)
vel0[-1] = 1.0
tmp = pathfollow07(v80_4, bp_beta, vel0, a, L, bp_V0, bp_lc, bp_M, N80, p80_4, nmax=1, intv=1, h=1e-3, hmin=1e-5,hmax=10.0, tol=1e-12)

# rho Re(f0) c h |b-Jv| (v,v0) residual num_iter


In [ ]:
bd80_p48_1 = pathfollow07(tmp[-1]['x'], bp_beta, vel0, a, L, bp_V0, bp_lc, bp_M, N80, p80_4, nmax=2000, intv=50, h=1e-3, hmin=1e-5,hmax=10.0, tol=1e-10, n_f=3, n_s=4)

In [ ]:
r = 2.5
MaxDist80 = []
for bd in bd80_p48_1:
    theta80 = fsolve(tobesolved, 60, args=(bd['x'], L, N80), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta80, L, N80), gargs=(L/N80, r, N80),
                               tmin=-r, tmax=N80-r, vectorized=True)
    MaxDist80.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist80_p48_1 = np.array(MaxDist80)

In [ ]:
np.save('ov_N80_p48_r2.5.npy', MaxDist80_p48_1)

In [ ]:
r = 4.0
MaxDist80 = []
for bd in bd80_p48_1:
    theta80 = fsolve(tobesolved, 60, args=(bd['x'], L, N80), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta80, L, N80), gargs=(L/N80, r, N80),
                               tmin=-r, tmax=N80-r, vectorized=True)
    MaxDist80.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist80_p48_2 = np.array(MaxDist80)

In [ ]:
np.save('ov_N80_p48_r4.0.npy', MaxDist80_p48_2)

In [ ]:
r = 8.0
MaxDist80 = []
for bd in bd80_p48_1:
    theta80 = fsolve(tobesolved, 60, args=(bd['x'], L, N80), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta80, L, N80), gargs=(L/N80, r, N80),
                               tmin=-r, tmax=N80-r, vectorized=True)
    MaxDist80.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist80_p48_3 = np.array(MaxDist80)

In [ ]:
np.save('ov_N80_p48_r8.0.npy', MaxDist80_p48_3)

In [ ]:
r = 12.0
MaxDist80 = []
for bd in bd80_p48_1:
    theta80 = fsolve(tobesolved, 60, args=(bd['x'], L, N80), xtol=1e-12)
    test = max_norm_difference(u_profile_optimized_with_shift, u0,
                               fargs=(bd['x'], theta80, L, N80), gargs=(L/N80, r, N80),
                               tmin=-r, tmax=N80-r, vectorized=True)
    MaxDist80.append([float(bd['a']), test['max_norm'], test['argmax'], test['converged']])

In [ ]:
MaxDist80_p48_4 = np.array(MaxDist80)

In [ ]:
np.save('ov_N80_p48_r12.0.npy', MaxDist80_p48_4)